In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/sample_data/IMDB Dataset.csv")

In [ ]:
df.shape

(50000, 2)

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

(49582, 2)

## Pre-processing

In [ ]:
import re

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


# Download required NLTK resources
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)


STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()


def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+", "", text)

    # Remove punctuation
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)

    # Remove HTML
    text = re.sub(r"<.*?>", "", text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [
        word for word in tokens
        if word not in STOP_WORDS
    ]

    # Stemming
    stemmed_words = [
        STEMMER.stem(token)
        for token in tokens
    ]

    return " ".join(stemmed_words)



In [ ]:
df["review"] = df["review"].apply(preprocess_text)


In [ ]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [ ]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


### 7. Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [ ]:
y = df["sentiment"]

In [ ]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [ ]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,1
1,wonder littl product br br film techniqu unass...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## Dataset & Data Loaders

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train.shape

(39665, 5000)

In [ ]:
X_test.shape

(9917, 5000)

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [ ]:
#def forward(self, x):

    #out, _ = self.rnn(x)

    #out = self.fc(out[:, -1, :])
    #return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.28688135743141174
epoch = 2/10 and loss = 0.20789417624473572
epoch = 3/10 and loss = 0.18166810274124146
epoch = 4/10 and loss = 0.12605014443397522
epoch = 5/10 and loss = 0.1854448914527893
epoch = 6/10 and loss = 0.2550050914287567
epoch = 7/10 and loss = 0.23191425204277039
epoch = 8/10 and loss = 0.18225610256195068
epoch = 9/10 and loss = 0.21272996068000793
epoch = 10/10 and loss = 0.2075589895248413


In [ ]:
# Evaluation

model.eval()

correct_vals = 0
tot_vals = 0

with torch.no_grad():

    for Xb, yb in test_loader:

        # Add sequence dimension
        Xb = Xb.unsqueeze(1)

        # Model prediction
        outputs = model(Xb)

        # Convert logits to probability
        probabilities = torch.sigmoid(outputs.squeeze())

        # Convert probability to 0 or 1
        predicted = (probabilities >= 0.5).float()

        # Count correct predictions
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()


# Calculate accuracy
accuracy = correct_vals / tot_vals

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 87.21%


###  It saves your trained RNN, This is the notebook cell you run after training.TF-IDF vectorizer, and LabelEncoder.

In [ ]:
import os
import torch
import joblib

os.makedirs("models", exist_ok=True)


# Save RNN model
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_size": input_size,
        "hidden_size": 128,
        "num_layers": 1
    },
    "models/rnn_sentiment_model.pth"
)


# Save TF-IDF vectorizer
joblib.dump(
    tf,
    "models/tfidf_vectorizer.pkl"
)


# Save label encoder
joblib.dump(
    le,
    "models/label_encoder.pkl"
)


print("All model files saved successfully!")

All model files saved successfully!


In [ ]:
# AFTER RUNING THIS FILE LOOK LIKE THIS
# models/
#    │ rnn_sentiment_model.pth
#    ├── tfidf_vectorizer.pkl
#    └── label_encoder.pkl



In [ ]:
import shutil

# Create models.zip from the models folder
shutil.make_archive("models", "zip", "models")

print("models.zip created successfully!")

models.zip created successfully!


In [ ]:
from google.colab import files

files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>